# Experimento — Correção Automatizada com e sem RAG (Livro de História)

Livro fonte: **História — Ensino Médio, 2ª Ed. (SEED-PR, 2007)**

**Fluxo:**
1. Setup (registro, login, turma e alunos)
2. Condição A — Com RAG (PDF + publish)
3. Condição B — Sem RAG
4. QA4 — Estabilidade (5 runs)
5. Exportação dos resultados

## 0. Configuração

In [ ]:
import requests, json, time
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path

BASE_URL = "http://localhost:8000"
PDF_PATH = Path("historia.pdf")   # mesmo diretório do notebook
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
assert PDF_PATH.exists(), f"PDF não encontrado em {PDF_PATH.resolve()}"

state = {}

def api(method, path, **kwargs):
    headers = kwargs.pop("headers", {})
    if "token" in state:
        headers["Authorization"] = f"Bearer {state['token']}"
    resp = getattr(requests, method)(f"{BASE_URL}{path}", headers=headers, **kwargs)
    if not resp.ok:
        print(f"ERRO {method.upper()} {path} -> {resp.status_code}: {resp.text[:300]}")
    return resp

def save_csv(df, name):
    path = OUTPUT_DIR / f"{name}_{TIMESTAMP}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"  Salvo: {path}")
    return path

print(f"Backend: {BASE_URL}")
print(f"PDF:     {PDF_PATH.resolve()}")
print(f"Output:  {OUTPUT_DIR.resolve()}")
print(f"Run ID:  {TIMESTAMP}")


## 1. Dados do Experimento

In [ ]:
# Unidade I  — Trabalho escravo e trabalho livre  (Folhas 1-5)
# Unidade II — Urbanização e industrialização      (Folhas 6-11)
# Unidade III— O Estado e as relações de poder     (Folhas 12-15)

QUESTIONS = [
    {"id": "Q1", "order": 1, "points": 10.0,
     "statement": ("(Unidade I) Explique a transição do trabalho escravo para o assalariado "
                   "no Brasil e nos EUA no séc. XIX, apontando semelhanças e diferenças no "
                   "contexto de consolidação do capitalismo.")},
    {"id": "Q2", "order": 2, "points": 10.0,
     "statement": ("(Unidade II) Analise a urbanização e industrialização no Brasil versus "
                   "Europa no séc. XIX. Discuta impactos sociais/econômicos/culturais e o papel "
                   "do Porto de Paranaguá na expansão capitalista no Paraná.")},
    {"id": "Q3", "order": 3, "points": 10.0,
     "statement": ("(Unidade III) Discuta como o Estado evoluiu da Antiguidade à modernidade. "
                   "Explique as relações de poder em Foucault e na Nova Esquerda Inglesa.")},
]

ANSWERS = {
    "Q1": {
        "A": ("A transição ocorreu de forma distinta no Brasil e nos EUA. Nos EUA, a abolição "
              "resultou da Guerra Civil (1861-1865). No Brasil foi gradual: Lei do Ventre Livre "
              "(1871), Lei dos Sexagenários (1885) e Lei Áurea (1888). Em ambos os países, "
              "libertos enfrentaram discriminação. O Brasil substituiu o escravo pelo imigrante "
              "europeu nas lavouras de café. Hobsbawm e Thompson destacam que o trabalho "
              "assalariado reconfigura, mas não elimina, as relações de exploração capitalista."),
        "B": ("No Brasil a escravidão acabou em 1888 (Lei Áurea). Nos EUA com a Guerra Civil "
              "em 1865. Em ambos, libertos ficaram marginalizados. No Brasil houve substituição "
              "pelo imigrante europeu; nos EUA os ex-escravizados permaneceram no sul sob "
              "outras formas de exploração."),
        "C": ("A escravidão acabou no Brasil em 1888 e nos EUA depois da guerra. Os ex-escravos "
              "passaram a receber salários. No Brasil vieram imigrantes para trabalhar."),
        "D": ("A escravidão acabou porque era errado. Depois todos passaram a ter salário. "
              "No Brasil foi em 1888."),
    },
    "Q2": {
        "A": ("A industrialização europeia do séc. XIX gerou êxodo rural e proletariado urbano. "
              "No Brasil ocorreu de forma tardia, ligada ao capital cafeeiro e à imigração. "
              "O Porto de Paranaguá foi central na integração regional ao mercado internacional, "
              "escoando erva-mate e madeira pela Estrada de Ferro Paranaguá-Curitiba. "
              "Enquanto na Europa a industrialização antecedeu a urbanização, no Brasil o "
              "processo foi invertido."),
        "B": ("Na Europa a industrialização gerou crescimento urbano e classe operária. No "
              "Brasil foi mais lento. O Porto de Paranaguá exportou erva-mate e madeira, "
              "integrando o Paraná ao capitalismo."),
        "C": ("A industrialização europeia veio da Revolução Industrial. No Brasil foi mais "
              "tarde. O Porto de Paranaguá ajudou o desenvolvimento do Paraná."),
        "D": ("A industrialização fez as cidades crescerem. O Porto de Paranaguá é importante "
              "para o Paraná."),
    },
    "Q3": {
        "A": ("Na Grécia e Roma o poder se legitimava por laços religiosos e militares; "
              "no feudalismo por vínculos de sangue. Com a modernidade surgem Estados "
              "nacionais baseados na soberania territorial (Weber: monopólio legítimo da "
              "violência). A Nova Esquerda Inglesa (Hobsbawm, Thompson) propõe analisar "
              "o poder pelas classes sociais. Foucault radicaliza com os 'micropoderes': "
              "o poder se exerce em escolas, prisões, hospitais — o biopoder."),
        "B": ("Na Grécia e Roma o Estado era diferente. Na Idade Média o poder era dos reis "
              "e da Igreja. Surgiram os Estados nacionais. Foucault mostrou que o poder "
              "está também nas instituições cotidianas. A Nova Esquerda analisava o poder "
              "pelas classes sociais."),
        "C": ("Na Grécia havia democracia. Na Idade Média os reis mandavam. Depois vieram "
              "os países modernos. Foucault falou sobre poder nas instituições."),
        "D": ("O Estado mudou muito ao longo da história. Hoje tem presidente e leis. "
              "Foucault estudou o poder."),
    },
}

STUDENTS = [
    {"full_name": "Ana Nivel A",   "email": "ana_a@escola.edu",   "level": "A"},
    {"full_name": "Bruno Nivel B", "email": "bruno_b@escola.edu", "level": "B"},
    {"full_name": "Carla Nivel C", "email": "carla_c@escola.edu", "level": "C"},
    {"full_name": "Diego Nivel D", "email": "diego_d@escola.edu", "level": "D"},
]

print(f"Dados: {len(QUESTIONS)} questoes, {len(STUDENTS)} alunos, "
      f"{len(QUESTIONS)*len(STUDENTS)} respostas")


## 2. Setup — Registro, Login, Turma e Alunos

In [ ]:
RUN_SUFFIX = TIMESTAMP[-6:]

# POST /users/register
teacher_email = f"prof.historia.{RUN_SUFFIX}@escola.edu"
teacher_password = "Historia@2024"
r = api("post", "/users/register", json={
    "name": "Prof. Historia", "email": teacher_email, "password": teacher_password
})
assert r.ok, f"Registro falhou: {r.text[:300]}"
print(f"Professor registrado: {teacher_email}")

# POST /auth/login  (body JSON, nao form-data)
r = api("post", "/auth/login", json={"email": teacher_email, "password": teacher_password})
assert r.ok, f"Login falhou: {r.text[:300]}"
state["token"] = r.json()["access_token"]
print("Login OK")

# POST /classes  -> resposta.uuid
r = api("post", "/classes", json={"name": f"Historia-{RUN_SUFFIX}", "year": 2026, "semester": 1})
assert r.ok, f"Turma falhou: {r.text[:300]}"
state["class_uuid"] = r.json()["uuid"]
print(f"Turma: {state['class_uuid']}")

# POST /classes/{uuid}/students  (cria e associa alunos)
payload = [{"full_name": s["full_name"], "email": f"{RUN_SUFFIX}.{s['email']}"} for s in STUDENTS]
r = api("post", f"/classes/{state['class_uuid']}/students", json={"students": payload})
assert r.ok, f"Alunos falhou: {r.text[:300]}"
print(f"{len(STUDENTS)} alunos adicionados")

# GET /classes/class/{uuid}  -> busca uuid de cada aluno
r = api("get", f"/classes/class/{state['class_uuid']}")
assert r.ok, f"Busca turma falhou: {r.text[:300]}"
email_to_uuid = {stu["email"]: stu["uuid"] for stu in r.json()["students"]}

state["students"] = []
for s in STUDENTS:
    prefixed = f"{RUN_SUFFIX}.{s['email']}"
    uuid = email_to_uuid.get(prefixed)
    assert uuid, f"UUID nao encontrado para {prefixed}"
    state["students"].append({**s, "uuid": uuid})
    print(f"  {s['full_name']} -> {uuid}")

print(f"\nSetup OK — {len(state['students'])} alunos")


## 3. Critérios de Avaliação Disponíveis

In [ ]:
# GET /grading-criteria — lista critérios pré-cadastrados no banco
r = api("get", "/grading-criteria")
assert r.ok, f"Criterios falhou: {r.text[:300]}"
all_criteria = r.json()

print(f"Criterios disponíveis ({len(all_criteria)}):")
for c in all_criteria:
    print(f"  [{c['code']}] {c['name']}")

# Seleciona todos ativos e distribui peso igual
SELECTED_CRITERIA = [c for c in all_criteria if c["active"]]
n = len(SELECTED_CRITERIA)
base_weight = round(1.0 / n, 4) if n > 0 else 1.0
for c in SELECTED_CRITERIA:
    c["weight"] = base_weight
    c["max_points"] = round(10.0 * base_weight, 2)

print(f"\n{n} criterios selecionados (peso={base_weight} cada)")


## 4. Função Principal do Experimento

In [ ]:
def run_condition(label, upload_pdf=True):
    print(f"\n{'='*60}\n  {label}\n{'='*60}")

    # 1. Cria exame — POST /exams
    r = api("post", "/exams", json={
        "title": f"Historia {label} {RUN_SUFFIX}",
        "class_uuid": state["class_uuid"],
        "status": "DRAFT",
    })
    assert r.ok, "Falha ao criar exame"
    exam_uuid = r.json()["uuid"]
    print(f"  Exame: {exam_uuid}")

    # 2. Upload PDF — POST /attachments/upload  (somente Condicao A)
    if upload_pdf:
        with open(PDF_PATH, "rb") as f:
            r = api("post", "/attachments/upload",
                    data={"exam_uuid": exam_uuid},
                    files={"file": (PDF_PATH.name, f, "application/pdf")})
        assert r.ok, "Falha upload PDF"
        print(f"  PDF: {PDF_PATH.name} ({PDF_PATH.stat().st_size/1024/1024:.1f} MB)")
        time.sleep(10)  # aguarda indexacao

    # 3. Questoes — POST /exam-questions
    question_uuids = {}
    for q in QUESTIONS:
        r = api("post", "/exam-questions", json={
            "exam_uuid": exam_uuid,
            "statement": q["statement"],
            "question_order": q["order"],
            "points": q["points"],
        })
        assert r.ok, f"Falha questao {q['id']}"
        question_uuids[q["id"]] = r.json()["uuid"]
    print(f"  {len(QUESTIONS)} questoes criadas")

    # 4. Criterios — POST /exam-criteria  (associa uuid de criterio pré-existente)
    for c in SELECTED_CRITERIA:
        r = api("post", "/exam-criteria", json={
            "exam_uuid": exam_uuid,
            "criteria_uuid": c["uuid"],
            "weight": c["weight"],
            "max_points": c["max_points"],
        })
        assert r.ok, f"Falha criterio {c['name']}"
    print(f"  {len(SELECTED_CRITERIA)} criterios associados")

    # 5. Publica — POST /exams/{exam_uuid}/publish
    r = api("post", f"/exams/{exam_uuid}/publish")
    assert r.ok, "Falha ao publicar"
    print("  Publicado")

    # 6. Respostas — POST /student-answers (uma por aluno x questao)
    for student in state["students"]:
        level = student["level"]
        for q in QUESTIONS:
            r = api("post", "/student-answers", json={
                "exam_uuid": exam_uuid,
                "question_uuid": question_uuids[q["id"]],
                "student_uuid": student["uuid"],
                "answer_text": ANSWERS[q["id"]][level],
            })
            assert r.ok, f"Falha resposta {q['id']} de {student['full_name']}"
        print(f"  {student['full_name']} (nivel {level}) — enviado")

    # 7. Aguarda correcao
    print("  Aguardando correcao automatica...")
    time.sleep(30)

    # 8. Resultados — GET /reviews/exams/{exam_uuid}
    r = api("get", f"/reviews/exams/{exam_uuid}")
    assert r.ok, "Falha ao buscar review"
    review = r.json()

    results = []
    for qr in review.get("questions", []):
        q_uuid = str(qr["question_uuid"])
        q_id = next((k for k, v in question_uuids.items() if v == q_uuid), q_uuid)
        for ans in qr.get("student_answers", []):
            s_uuid = str(ans["student_uuid"])
            student = next((s for s in state["students"] if s["uuid"] == s_uuid), {})
            results.append({
                "condition": label,
                "student": student.get("full_name", s_uuid),
                "level": student.get("level", "?"),
                "question": q_id,
                "score": ans.get("score"),
                "max_score": qr.get("max_score"),
                "status": ans.get("status"),
                "feedback_length": len(ans.get("feedback") or ""),
                "c1_score": ans.get("c1_score"),
                "c2_score": ans.get("c2_score"),
                "divergence": ans.get("divergence_detected", False),
                "exam_uuid": exam_uuid,
            })

    df = pd.DataFrame(results)
    scored = df[df["score"].notna()]
    media = scored["score"].mean() if not scored.empty else float("nan")
    print(f"\n  Media ({label}): {media:.2f} ({len(scored)}/{len(df)})")
    return {"df": df, "exam_uuid": exam_uuid}


## 5. Condição A — Com RAG

In [ ]:
cond_a = run_condition("Condicao A — Com RAG", upload_pdf=True)

## 6. Condição B — Sem RAG

In [ ]:
cond_b = run_condition("Condicao B — Sem RAG", upload_pdf=False)

## 7. Análise A vs B

In [ ]:
df_a = cond_a["df"].copy()
df_b = cond_b["df"].copy()

merged = df_a[["student","level","question","score"]].merge(
    df_b[["student","question","score"]], on=["student","question"], suffixes=("_A","_B"))
merged["Delta"] = merged["score_A"] - merged["score_B"]

print("── Por aluno/questao ──")
print(merged.to_string(index=False))
print(f"\nDelta medio (A-B): {merged['Delta'].mean():+.2f}")
print("\nPor questao:")
print(merged.groupby("question")["Delta"].mean().to_string())

summary = merged.groupby("level").agg(
    media_A=("score_A","mean"), media_B=("score_B","mean"), delta=("Delta","mean")
).reset_index()
print("\n── Por nivel ──")
print(summary.to_string(index=False))

comparison = merged
save_csv(df_a,       "historia_cond_A")
save_csv(df_b,       "historia_cond_B")
save_csv(comparison, "historia_comparacao_A_vs_B")
save_csv(summary,    "historia_resumo_por_nivel")


## 8. QA4 — Estabilidade (5 execuções)

In [ ]:
QA4_RUNS = 5
qa4_results = []
for i in range(QA4_RUNS):
    print(f"\nRun {i+1}/{QA4_RUNS}")
    result = run_condition(f"QA4 Run {i+1}", upload_pdf=True)
    qa4_results.append({"run": i+1, "exam_uuid": result["exam_uuid"], "df": result["df"]})
    time.sleep(5)
print(f"\nQA4 concluido — {QA4_RUNS} runs")


## 9. Análise de Estabilidade

In [ ]:
qa4_all = pd.concat([r["df"].assign(run=r["run"]) for r in qa4_results], ignore_index=True)

qa4_stats = qa4_all.groupby(["student","question"]).agg(
    mean_score=("score","mean"), var_score=("score","var"),
    std_score=("score","std"), min_score=("score","min"), max_score=("score","max")
).reset_index()
qa4_stats["var_abs"] = qa4_stats["max_score"] - qa4_stats["min_score"]
print(qa4_stats.to_string(index=False))

qa4_level = qa4_all.groupby(["level","question"]).agg(
    mean_score=("score","mean"), std_score=("score","std"),
    var_abs=("score", lambda x: x.max()-x.min())
).reset_index()
print("\n── Por nivel ──")
print(qa4_level.to_string(index=False))

save_csv(qa4_all,   "historia_qa4_todas_runs")
save_csv(qa4_stats, "historia_qa4_estatisticas")
save_csv(qa4_level, "historia_qa4_resumo_por_nivel")

qa4_path = OUTPUT_DIR / f"historia_qa4_{TIMESTAMP}.json"
with open(qa4_path, "w", encoding="utf-8") as f:
    json.dump({"label":"QA4 Historia","num_runs":QA4_RUNS,"timestamp":TIMESTAMP,
               "runs":[{"run":r["run"],"exam_uuid":r["exam_uuid"]} for r in qa4_results]},
              f, ensure_ascii=False, indent=2, default=str)
print(f"  JSON: {qa4_path}")


## 10. Resumo Final

In [ ]:
print("="*70)
print("  RESUMO — HISTORIA (SEED-PR, 2a Ed.)")
print("="*70)
print(f"  PDF:    {PDF_PATH.name} ({PDF_PATH.stat().st_size/1024/1024:.1f} MB, 400 pags)")
print(f"  Run ID: {TIMESTAMP}")
print()
print(f"  Condicao A (com RAG):  media = {cond_a['df']['score'].mean():.2f}")
print(f"  Condicao B (sem RAG):  media = {cond_b['df']['score'].mean():.2f}")
print(f"  Delta medio (A-B):     {comparison['Delta'].mean():+.2f}")
print(f"\n  QA4 — {QA4_RUNS} execucoes:")
print(f"    Variancia media:          {qa4_stats['var_score'].mean():.2f}")
print(f"    Desvio padrao medio:      {qa4_stats['std_score'].mean():.2f}")
print(f"    Variacao absoluta maxima: {qa4_stats['var_abs'].max():.2f}")
qa4_ok = (qa4_stats["var_abs"] <= 1.0).all()
print(f"    QA4 aprovado (<=1.0):     {'SIM' if qa4_ok else 'NAO'}")
print(f"\n  Output: {OUTPUT_DIR.resolve()}/")
print("="*70)
